##### ============= STAGE 06 FINANCIAL IMPACT FOR THE OUTAGE IN THE REGION ============== ########

From the INSIGHT ANALYSIS, I was able to established:

- `Grain: daily = one row per customer‑day`

###### `Columns (simplified):`

- customer_id

- region

- tariff_plan ('fixed' / 'variable')

- is_smart_meter

- date

- daily_kwh

- daily_outage_minutes

- daily_bill_eur

- any_complaint (0/1)

- high_outage_day (0/1, e.g. daily_outage_minutes >= 30)

Established that:

In the North region, on high‑outage days, both fixed tariff and variable‑tariff customers have 100% complaint rate. With this, It is important to study the financial implications of these shortcoming and I would aim to answer the question: `What is the financial impact of this problem?`.

I would estimate the following to answer the question:

- Cost per complaint (operational cost)

- Potential churn risk and lost margin

- A simple annualised impact estimate.

In [2]:
import numpy as np
import pandas as pd
import os

In [3]:
# LOAD DATA FROM DATA/CLEANED FOLDER
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
data_path = os.path.join(BASE_DIR, "data", "engineered", "energy_engineered_daily_aggregated_data.csv")
daily_df = pd.read_csv(data_path)
daily_df.head()

,customer_id,region,tariff_plan,is_smart_meter,date,day_name,daily_kwh,daily_outage_minutes,daily_bill_eur,any_complaint,high_outage_day
0,201,East,fixed,1,2026-01-01,Thursday,40.674543,15,7.321418,0,1
1,201,East,fixed,1,2026-01-02,Friday,29.456769,16,5.302218,0,1
2,201,East,fixed,1,2026-01-03,Saturday,32.690122,15,5.884222,1,1
3,201,East,fixed,1,2026-01-04,Sunday,34.040236,13,6.127243,1,0
4,201,East,fixed,1,2026-01-05,Monday,31.587888,17,5.685820,1,1


In [4]:
daily_df.columns

Index(['customer_id', 'region', 'tariff_plan', 'is_smart_meter', 'date',
       'day_name', 'daily_kwh', 'daily_outage_minutes', 'daily_bill_eur',
       'any_complaint', 'high_outage_day'],
      dtype='str')

###### `FOCUS ON THE NORTH SINCE IT IS THE REGION WITH STRONGEST EFFECT FOR BOTH FIXED AND VARIABLE TARIFF`

In [5]:
NORTH_DAILY = daily_df[daily_df["region"] == "North"].copy()
NORTH_DAILY.head()

,customer_id,region,tariff_plan,is_smart_meter,date,day_name,daily_kwh,daily_outage_minutes,daily_bill_eur,any_complaint,high_outage_day
85,202,North,variable,1,2026-01-01,Thursday,35.050295,24,5.888450,1,1
86,202,North,variable,1,2026-01-02,Friday,30.755358,22,5.166900,1,1
87,202,North,variable,1,2026-01-03,Saturday,44.236458,23,8.246287,1,1
88,202,North,variable,1,2026-01-04,Sunday,37.898177,22,6.366894,1,1
89,202,North,variable,1,2026-01-05,Monday,40.381781,22,6.784139,0,1


In [6]:
NORTH_DAILY[['region', 'tariff_plan', 'high_outage_day', 'any_complaint']].head()

,region,tariff_plan,high_outage_day,any_complaint
85,North,variable,1,1
86,North,variable,1,1
87,North,variable,1,1
88,North,variable,1,1
89,North,variable,1,0


###### `Estimate incremental complaint rate and volume`

We want to quantify:

Complaint rate for fixed vs variable on high‑outage days

The incremental complaints attributable to variable tariff under outages

In [7]:
complaint_summary = (
    NORTH_DAILY
    .groupby(['region', 'tariff_plan', 'high_outage_day'])
    .agg(
        no_of_days=('customer_id', 'count'),
        complaint_days=('any_complaint', 'sum')
    )
    .reset_index()
)

complaint_summary['complaint_rate'] = round(
    complaint_summary['complaint_days'] / complaint_summary['no_of_days'],
    2
)

complaint_summary

,region,tariff_plan,high_outage_day,no_of_days,complaint_days,complaint_rate
0,North,fixed,1,720,466,0.65
1,North,variable,1,540,451,0.84


Let’s extract the key numbers for `high‑outage days only:`

In [8]:
high_outage = complaint_summary[complaint_summary['high_outage_day'] == 1].copy()

fixed_rate = high_outage.loc[high_outage['tariff_plan'] == 'fixed', 'complaint_rate'].iloc[0]
variable_rate = high_outage.loc[high_outage['tariff_plan'] == 'variable', 'complaint_rate'].iloc[0]

rate_uplift = variable_rate - fixed_rate
uplift_factor = variable_rate / fixed_rate

fixed_rate, variable_rate, rate_uplift, uplift_factor
print(f"fixed_rate = {fixed_rate}, \nvariable_rate = {variable_rate}, \nrate_uplift = {rate_uplift}, \nuplift_factor = {uplift_factor}")

fixed_rate = 0.65, 
variable_rate = 0.84, 
rate_uplift = 0.18999999999999995, 
uplift_factor = 1.2923076923076922


How many `extra complaints` does that represent?

In [9]:
# NUMBER OF VARIABLE-TARIFF CUSTOMER-DAYS ON HIGH-OUTAGE DAYS
var_high = NORTH_DAILY[
    (NORTH_DAILY['tariff_plan'] == 'variable') &
    (NORTH_DAILY['high_outage_day'] == 1)
]

n_var_high_days = len(var_high)
expected_complaints_if_fixed = n_var_high_days * fixed_rate
actual_complaints_variable = var_high['any_complaint'].sum()

incremental_complaints = actual_complaints_variable - expected_complaints_if_fixed

n_var_high_days, expected_complaints_if_fixed, actual_complaints_variable, incremental_complaints

print(f"No_of_variable_high_outage_days = {n_var_high_days}\nExpected_complaints_if_fixed = {expected_complaints_if_fixed}\nActual_complaints_variable = {actual_complaints_variable}\nIncremental_complaints_due_to_variable_tariff = {incremental_complaints}")

No_of_variable_high_outage_days = 540
Expected_complaints_if_fixed = 351.0
Actual_complaints_variable = 451
Incremental_complaints_due_to_variable_tariff = 100.0


##### `COST PER COMPLAINT`
- **Making explicit assumptions that, the cost per complain (HR, contact center, back office, QA etc) is:**

        - `Cost_Per_Complaint = 15 euros`
- **Convert the 60 days of data into yearly (annualizing to 365 days)**

In [10]:
cost_per_complaint = 15.0

# INCREMENTAL COST OVER THE OBSERVED PERIOD
incremental_cost_period = incremental_complaints * cost_per_complaint

# SCALE TO ANNUAL - SIMPLE PROPORTIONAL SCALING
days_observed = NORTH_DAILY['date'].nunique()
annual_factor = 365 / days_observed

incremental_cost_annual = incremental_cost_period * annual_factor

incremental_cost_period, incremental_cost_annual
print(f"Incremental_cost_over_observed_period = €{incremental_cost_period:.2f}\nEstimated_annual_incremental_cost = €{incremental_cost_annual:.2f}")

print("======================================")
print("This shows Incremental complaint handling cost over the 60-day window and a rough annualised cost in euros.")

Incremental_cost_over_observed_period = €1500.00
Estimated_annual_incremental_cost = €9125.00
This shows Incremental complaint handling cost over the 60-day window and a rough annualised cost in euros.


##### ESTIMATE CHURN RISK AND LOST MARGIN
I delved deeper to investigate the churn with the extra complaints:

`What if some of these extra complaints turn into churn?`

**Assuming the following:**

- Baseline annual churn probability for this segment -> p_churn_base = 10%

- Incremental churn probability for customers with at least one complaint -> delta_p_churn = 5%

- Average annual gross margin per customer -> avr_gross_margin_annual = 120 EUR.

**To estimate the following:**

- How many unique customers are in the 'variable, high‑outage, complaint' segment.

- How many extra churned customers that implies.

- The lost margin from that churn.

In [11]:
# CUSTOMER IN THE RISK SEGMENT: VARIABLE TARRIF, HIGH-OUTAGE DAYS WITH COMPLAINTS
risk_segment = NORTH_DAILY[
    (NORTH_DAILY['tariff_plan'] == 'variable') &
    (NORTH_DAILY['high_outage_day'] == 1) &
    (NORTH_DAILY['any_complaint'] == 1)
]

n_unique_risk_customers = risk_segment['customer_id'].nunique()
n_unique_risk_customers

9

In [12]:
p_churn_base = 0.10 
delta_p_churn = 0.05  
avr_gross_margin_annual = 120.0    

# Expected extra churned customers in this segment
extra_churned_customers = n_unique_risk_customers * delta_p_churn

# Lost margin from extra churn
lost_margin_annual = extra_churned_customers * avr_gross_margin_annual

extra_churned_customers, lost_margin_annual
print(f"Estimated_extra_churned_customers_due_to_variable_tariff = {extra_churned_customers:.1f}\nEstimated_annual_lost_margin_due_to_extra_churn = €{lost_margin_annual:.2f}")
print("==========================================")
print("This gives a second financial lever including Operational cost of handling extra complaints and a Strategic cost of losing customers (lost margin_annual) due to higher churn in the risk segment.")


Estimated_extra_churned_customers_due_to_variable_tariff = 0.5
Estimated_annual_lost_margin_due_to_extra_churn = €54.00
This gives a second financial lever including Operational cost of handling extra complaints and a Strategic cost of losing customers (lost margin_annual) due to higher churn in the risk segment.


##### `FULL FINANCIAL IMPACT SUMMARY`

In [15]:
impact_summary = {
    'days_observed': days_observed,
    'incremental_complaints_period': float(incremental_complaints),
    'incremental_complaint_cost_period_eur': float(incremental_cost_period),
    'incremental_complaint_cost_annual_eur': float(incremental_cost_annual),
    'risk_segment_customers': int(n_unique_risk_customers),
    'extra_churned_customers_est': float(extra_churned_customers),
    'lost_margin_annual_eur': float(lost_margin_annual),
    'total_annual_impact_eur': float(incremental_cost_annual + lost_margin_annual)
}

pd.DataFrame([impact_summary]).T.rename(columns={0: 'value (EUR)'})


,value (EUR)
days_observed,60.00
incremental_complaints_period,100.00
incremental_complaint_cost_period_eur,1500.00
incremental_complaint_cost_annual_eur,9125.00
risk_segment_customers,9.00
extra_churned_customers_est,0.45
lost_margin_annual_eur,54.00
total_annual_impact_eur,9179.00


##### **In the North region, fixed-tariff and variable‑tariff customers on high‑outage days generate about `65%` and `84%` complaint rate respectively.** 

##### **Over a year, this translates into roughly `€9.1k` in extra complaint‑handling cost and an estimated `€54` in lost margin from churn, for a total impact of around `€9–10k` per year in this segment alone. Consequently, If similar patterns exist in other regions or tariff types, the group‑wide impact could be multiples of this figure.**